# Contextual Embeddings Explorer: "layer" across ML/AI, Science, Cooking, and Skincare

This project uses a pretrained BERT model to explore how the word **"layer"** takes on different meanings across domains:
- **ML/AI**: *hidden layer*, *attention layer*, *embedding layer*
- **Science**: *ozone layer*, *sedimentary layer*, *atmospheric layer*
- **Cooking**: *layer a lasagna*, *layer the cake*, *pastry layer*
- **Skincare**: *layer a serum*, *protective layer*, *apply in layers*

By extracting contextual embeddings with BERT, we can visualize how the same word clusters differently in vector space depending on its surrounding context.

In [ ]:
# install all required dependencies into the current notebook's Python environment.
# run this cell first before anything else.
import sys
!{sys.executable} -m pip install -r requirements.txt

In [ ]:
import requests # makes https requests to Wikipedia API
import re # regex -> split article text into sentences 

TARGET_WORD = "layer" # word "layer" -> we want to understand its contextual meaning
WORD_VARIANTS = {"layer", "layers", "layered", "layering", "layerwise"} # different forms of the word
HEADERS = {"User-Agent": "contextual-embeddings-project/1.0"} # user-agent header needed for Wikipedia API -> identify who makes the request


# function finds sentences that contain "layer"
def sentences_from_text(text, domain):
    results = [] # list will gather all matched sentence records

    # perform regex to get block of article texts into individual sentences
    for sent in re.split(r'(?<=[.!?])\s+', text):
        words = sent.strip().split() # get rid of leading/trailing whitespaces
        clean = [w.lower().strip('.,!?;:"\'()[]{}') for w in words]

        for i, w in enumerate(clean):

            # do any words match our word variants?
            # sentence must be at least four words too
            if w in WORD_VARIANTS and len(words) >= 4:
                results.append({"word": clean[i], "sentence": clean, "domain": domain})
                break
    return results


# function downloads one Wikipedia article
def fetch_wikipedia(title, domain):

    # try to get Wikipedia's plain text of an article
    resp = requests.get(
        "https://en.wikipedia.org/w/api.php",
        params={"action": "query", "titles": title, "prop": "extracts", "explaintext": True, "format": "json"},
        headers=HEADERS
    )

    # request failed, so we skip it
    if not resp.ok or not resp.text.strip():
        print(f"  [skip] {title}: HTTP {resp.status_code}")
        return []
    
    # if request succeeds, pass text to sentenced_from_text function and get those results
    pages = resp.json()["query"]["pages"]
    text = next(iter(pages.values())).get("extract", "")
    return sentences_from_text(text, domain)

# list where all sentences will be collected
# each item is a dictionary containing -> word, sentence, domain
# EXAMPLE
#     "word": "layer",
#     "sentence": ["the", "ozone", "layer", "absorbs", "most", ...],
#     "domain": "science"
items = [] 

# ALL for loops will go through a list of Wikipedia articles for that specific domain
# will fetch sentences and add them to items

# ML/AI — layer as neural network component
for title in ["Artificial neural network", "Deep learning", "Transformer (machine learning model)", "Convolutional neural network", "Recurrent neural network"]:
    items.extend(fetch_wikipedia(title, "ml_ai"))

# Science — layer as physical/natural stratum
for title in ["Ozone layer", "Atmosphere of Earth", "Sedimentary rock", "Stratum", "Earth's mantle", "Ocean", "Rainforest"]:
    items.extend(fetch_wikipedia(title, "science"))

# Cooking — layer as arranging food in stacks
for title in ["Lasagne", "Layer cake", "Baklava", "Mille-feuille", "Trifle", "Tiramisu", "Moussaka"]:
    items.extend(fetch_wikipedia(title, "cooking"))

# Skincare — layer as applying product over product
for title in ["Skin care", "Skincare", "Sunscreen", "Makeup", "Skin", "Dermatology", "Skin", "Theatrical makeup", "Stage makeup", "Wound healing", "Epidermis", "Cosmeceutical"]:
    items.extend(fetch_wikipedia(title, "skincare"))

# print how many sentences we collected in total
from collections import Counter
print(f"Total sentences: {len(items)}")
print(Counter(item['domain'] for item in items))

In [ ]:
# preview a few sentences from each domain
from itertools import groupby

# loop over each domain
for domain in ['ml_ai', 'science', 'cooking', 'skincare']:

    # filter items down to each domain
    domain_items = []

    # items is list of dictionaries
    # each item is a dictionary -> word, sentence, domain
    for item in items:
        if item['domain'] == domain:
            domain_items.append(item)


    # print three sentences for each domain we collected
    # gives us some understanding of what kinds of sentences using our word variants
    print(f"\n{domain.upper()} ({len(domain_items)} sentences)\n----------------------------------")
    for item in domain_items[:3]:
        print(f"[{item['word']}] {' '.join(item['sentence'][:15])}...")

In [ ]:
import torch # helps us work with tensors, BIG matrices/dimensions
from transformers import BertModel, AutoTokenizer # loads pretrained BERT model and tokenizer

In [ ]:
# if available, use gpu
# not available... use cpu
device = "cuda" if torch.cuda.is_available() else "cpu" # i'm on mac, so using cpu

In [ ]:
# may get a warning in the LOAD REPORT. This is ok!
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased") # converts raw text into numbers BERT can read
model = BertModel.from_pretrained("bert-base-uncased") # actual pretrained BERT NN

In [ ]:
# function turns a batch of sentences into numbers BERT can read
# also records where the word "layer" or variants appears in the sentence
def tokenize(batch):

    # stores position of target word in each sentence
    token_indices = []

    # tokenize all sentences at once
    # is_split_into_words -> tells tokenizer the sentences are already split into word lists
    encoded_input = tokenizer(batch['sentence'], is_split_into_words=True)

    # loops through each sentence in the batch
    for i in range(len(batch['sentence'])):

      # finds the position of "layer" in original word list for this sentence
      word_index = batch['sentence'][i].index(batch['word'][i])

      # map that word position to its position in the tokenized version
      # BERT may split one word into multiple tokens, so positions can shift
      token_span = encoded_input.word_to_tokens(i, word_index)

      if token_span is not None:
        # save position of the first token for "layer"
        token_indices.append(token_span.start)
      else:
        # if word couldn't be found in the tokens, mark it as missing
        token_indices.append(None)

    # will eventually return full batch
    # turn into one dictionary to return
    output = {
      "input_ids": encoded_input["input_ids"],
      "token_type_ids": encoded_input["token_type_ids"],
      "attention_mask": encoded_input["attention_mask"],
      "token_indices": token_indices
    }

    # check that every sentence has valid position for layer
    assert not any(x is None for x in output["token_indices"]), "Target token not found in sentence!"
    
    # check that number of positions matches number of sentences
    assert len(token_indices) == len(batch["sentence"]), "Token indices is the wrong length!"

    return output

In [ ]:
# adds pads to shorter sentences
# ensures that all sentences in a batch are the same length
from torch.nn.utils.rnn import pad_sequence 

# function prepares a batch of sentences as tensors for the model
def collate(items, device="cpu"):

    batch = [] # will become main dictionary
    keys = items[0].keys() # get all keys from list (word, sentence, domain)

    # turn list of key value pairs into one dictionary
    # items is a list!!! not dictionary
    batch = {key: [item[key] for item in items] for key in keys}

    # run tokenization on the whole batch
    # converts sentences to numbers and finds the target words' positions
    tokenized_batch = tokenize(batch)

    outputs = {
        # adds pad to sentences to make it all the same length, then converted to tensor
        'input_ids': pad_sequence([torch.tensor(x) for x in tokenized_batch['input_ids']], batch_first=True).to(device),
        #these are all zeros since we only have one sentence per input
        'token_type_ids': pad_sequence([torch.tensor(x) for x in tokenized_batch['token_type_ids']], batch_first=True).to(device),
        # 1 -> real token
        # 0 -> padding
        # helps BERT understand what to ignore 
        'attention_mask': pad_sequence([torch.tensor(x) for x in tokenized_batch['attention_mask']], batch_first=True).to(device),
        # position of "layer" in each sentence
        # will be useful later to extract just that word's embedding
        'token_indices': torch.tensor(tokenized_batch['token_indices']).to(device)
    }

    # now it is ready to be passed into BERT to get actual contextual meaning
    return outputs


In [ ]:
# function will extract the embedding vector for the word "layer" from BERT's output
# batch_reps should have shape (B, D)
# where B is the batch size and D is the dimension of the hidden state
# (for BERT, D = 768)
def get_token_embedding(model_output, token_indices, layer=-1):

    # grab hidden states from a specific BERT layer
    # layer=-1 is the last layer, is the most context-aware representation
    hidden_state = model_output.hidden_states[layer]

    # get number of sentences in this batch
    batch_size = hidden_state.shape[0]

    # creates [0,1,2,...,B-1] to index each sentence individually
    batch_indices = torch.arange(batch_size)

    # pick out the embedding at the "layer" token position for each sentence
    # shape: (B, 768)
    # one 768-number vector per sentence
    batch_reps = hidden_state[batch_indices, token_indices]

    # detach from computation graph (NO GRADIENTS NEEDED) 
    # move to CPU for storage
    return batch_reps.detach().cpu()


In [ ]:
from torch.utils.data import DataLoader # handles splitting data into batches and feeding them to the model
from tqdm import tqdm # displays progress bar so we see how far along inference is

# function runs BERT on all the data in batches and yields each batch with its model output
def iter_outputs(data, model, batch_size=128):
    model.eval()  # setting eval mode disables dropout (and other stuff)
    model.to(device)  # we put the model on the GPU or CPU

    dataloader = DataLoader(
        data, # full list of sentence items
        batch_size=batch_size, # how many sentences to process at once
        shuffle=False, # keep original order so embeddings stay aligned with items
        collate_fn=collate  # we pass in our collate function to tokenize and pad each batch
    )

    # disable gradient calculation -> we aren't training / backpropagating
    # this saves memory and speeds things up
    with torch.no_grad():

        # loop over each batch with a progress bar
        for batch in tqdm(dataloader):
            output = model(
                input_ids=batch["input_ids"].to(device), # token IDS for each sentence
                attention_mask=batch["attention_mask"].to(device), # tells BERT which tokens are real vs. padding
                token_type_ids=batch["token_type_ids"].to(device), # segment labels (all zeros here)

                # by default, this only returns the last layer hidden states
                # we want the flexibility to look at other layers, so we set
                # `output_hidden_states=True`
                output_hidden_states=True # returns hidden states from all 12 BERT layers, not just the last
            )

            # yield the batch and output together so the caller can extract embeddings
            yield batch, output


In [ ]:
# function runs BERT on all sentences and collects the "layer" embedding for each one
def get_all_embeddings():
    # list of all embeddings from each batch
    embeddings = []

    # loop through all batches and their model outputs
    for batch, batch_output in iter_outputs(items, model):

        # extract the "layer" token embedding for each sentence in this batch
        # add it to the list
        embeddings.append(get_token_embedding(batch_output, batch["token_indices"]))

    # stack all batches into one matrix
    # final shape: (459, 768) 
    # one 768-number vector per sentence
    # there are 459 sentences in total
    return torch.concat(embeddings, dim=0)

# run the function and store all embeddings
embeddings = get_all_embeddings()

## Exploring contextual embeddings

With contextual embeddings, the representations change depending on the context; let's see that in action by looking at the sentences where the target word embeddings have the greatest similarity.

In [ ]:
# find the k most similar embeddings to a given vector using cosine similarity
# cosine similarity measures how similar two vectors are -> look at their vector orientation
# does not matter the size of the vectors
def nearest_neighbors(vec, matrix, k=10):

    # compute cosine similarity between vector and every row in matrix
    # result is [-1, 1] -> this is orientation of how similar they are
    # 1 = most similar
    cos_sim = ((vec @ matrix.T) / (vec.norm() * matrix.T.norm(dim=0, keepdim=True))).squeeze()
    
    # sort by highest similarity
    # negate so argsort goes from highest to lowest
    # take top k
    inds = torch.argsort(-cos_sim)[:k]

    # return the indices of the top k sentences and their similarity scores
    return inds, cos_sim[inds]

# function prints the query setence and its nearest neighbors in embedding space
def show_nearest_neighbor_sentences(index, embeddings):

    # print the query sentence and which form of "layer" was matched
    print(f"QUERY (target word={items[index]['word']}):")
    print(" ".join(items[index]["sentence"]))

    # find the 5 most similar embeddings to the query
    print("NEIGHBORS:")
    inds, _ = nearest_neighbors(embeddings[index], embeddings, k=6)

    # print each neighbor sentence
    for ind in inds:
        print("-", " ".join(items[ind]["sentence"]))

In [ ]:
# show sentence 450 and its 5 nearest neighbors to demonstrate domain specific clustering
show_nearest_neighbor_sentences(450, embeddings)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np
from sklearn.manifold import TSNE

# Run t-SNE on embeddings
# right now, embeddings are 768 dimensions -> turn into 2D vector so we can plot it into 2D space
tsne = TSNE(n_components=2, random_state=42, perplexity=30)
reduced = tsne.fit_transform(embeddings.cpu().numpy())

# Build domain labels aligned to items
domains = [item["domain"] for item in items]
unique_domains = sorted(set(domains))
colors = cm.tab10(np.linspace(0, 1, len(unique_domains)))
domain_to_color = {d: c for d, c in zip(unique_domains, colors)}

# plot
fig, ax = plt.subplots(figsize=(10, 7))
for domain in unique_domains:
    idxs = [i for i, d in enumerate(domains) if d == domain]
    ax.scatter(
        reduced[idxs, 0], reduced[idxs, 1],
        c=[domain_to_color[domain]],
        label=domain, alpha=0.7, s=40
    )

ax.legend(title="Domain", bbox_to_anchor=(1.05, 1), loc="upper left")
ax.set_title('t-SNE of BERT Contextual Embeddings for "layer"')
ax.set_xlabel("t-SNE Dimension 1")
ax.set_ylabel("t-SNE Dimension 2")
plt.tight_layout()
plt.savefig("layer_tsne.png", dpi=150)
plt.show()

print("Distinct Clusters (Semantic Separation):")
print("  Notice the cooking cluster on the far left and ML/AI at the top.")
print("  The distance between these groups shows that BERT's representation of 'layering a cake' is mathematically very different from 'layering a neural network'.")
print()
print("Semantic Sub-clusters:")
print("  Within a single domain (like Science), you may see multiple smaller islands.")
print("  This likely reflects different sub-contexts — e.g. 'sedimentary layers' (geological) vs. 'layering samples' (biological/lab).")
print()
print("Overlapping Regions (Semantic Bleed):")
print("  Where Science and Skincare overlap, this makes sense — 'a layer of skin' (skincare) and 'a layer of tissue' (science/biology) are conceptually close.")
print()
print("Tight ML/AI Cluster:")
print("  ML/AI clusters tightly and separately because its surrounding vocabulary (weights, gradient, attention, hidden) is highly specialized.")
print("  BERT has no trouble isolating this sense from all others.")

## Findings & Analysis

This project used BERT's contextual embeddings to explore how the word **"layer"** takes on different meanings across four domains: ML/AI, Science, Cooking, and Skincare.

### Cluster Separation
The t-SNE visualization shows that **ML/AI** forms the most isolated and tightly packed cluster. This is expected — the vocabulary surrounding "layer" in ML contexts (*hidden, attention, weights, gradient*) is highly specialized and unlike any everyday usage. Cooking also forms a distinct cluster, reflecting that the culinary sense of "layer" (*lasagna, pastry, bake*) shares little with the technical domains.

### Semantic Overlap
**Science and Skincare** show the most overlap. This makes linguistic sense: both domains use "layer" in a biological or physical context — *a layer of tissue* and *a layer of skin* are semantically close even across domains. BERT picks up on this similarity through the shared surrounding vocabulary.

### Sub-clusters Within Domains
Within Science, smaller sub-clusters emerge — likely separating geological uses (*sedimentary layer, rock layer*) from atmospheric ones (*ozone layer, atmospheric layer*). This shows that BERT captures fine-grained semantic distinctions even within a single domain label.

### Key Takeaway
BERT does not treat "layer" as a single fixed concept. Instead, its representation shifts based on surrounding context, producing embeddings that reflect the domain-specific meaning of the word. The nearest-neighbor results confirm this: sentences about neural network layers retrieve other ML sentences, while sentences about the ozone layer retrieve science sentences — even though the surface form of the word is identical.